In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2023_Jahangirpuri_Delhi_DPCC_2023.xlsx")

In [4]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,303.0,183.0,187.0,115.0,100.0,112.0,123.0,NaN,153.0,161.0,384.0,378.0
1,2,374.0,224.0,260.0,179.0,85.0,178.0,58.0,NaN,156.0,169.0,422.0,389.0
2,3,402.0,210.0,191.0,198.0,156.0,101.0,109.0,78.0,156.0,189.0,484.0,342.0
3,4,372.0,302.0,169.0,114.0,187.0,196.0,162.0,96.0,147.0,198.0,436.0,333.0
4,5,375.0,262.0,208.0,188.0,214.0,168.0,91.0,NaN,112.0,217.0,473.0,316.0
5,6,404.0,359.0,181.0,201.0,282.0,160.0,58.0,114.0,118.0,256.0,461.0,321.0
6,7,395.0,325.0,194.0,219.0,227.0,258.0,68.0,123.0,115.0,247.0,407.0,370.0
7,8,400.0,187.0,270.0,230.0,153.0,180.0,57.0,NaN,100.0,164.0,455.0,373.0
8,9,438.0,246.0,139.0,318.0,232.0,214.0,58.0,132.0,42.0,183.0,458.0,365.0
9,10,430.0,194.0,234.0,267.0,245.0,166.0,NaN,140.0,33.0,NaN,316.0,352.0


In [5]:
df.info()
df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        40 non-null     object 
 1   January    35 non-null     float64
 2   February   33 non-null     float64
 3   March      35 non-null     float64
 4   April      34 non-null     float64
 5   May        36 non-null     float64
 6   June       34 non-null     float64
 7   July       23 non-null     float64
 8   August     30 non-null     float64
 9   September  34 non-null     float64
 10  October    37 non-null     float64
 11  November   34 non-null     float64
 12  December   34 non-null     float64
dtypes: float64(12), object(1)
memory usage: 4.3+ KB


(41, 13)

In [6]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [8]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [9]:
# Define a function for outlier handling
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            # Replace outliers with mean
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [10]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready.head()

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,303.0,183.0,187.0,115.0,100.0,112.0,71.043478,114.033333,153.0,161.0,384.0,378.0
1,2,374.0,224.0,260.0,179.0,85.0,178.0,58.000000,114.033333,156.0,169.0,422.0,389.0
2,3,402.0,210.0,191.0,198.0,156.0,101.0,71.043478,78.000000,156.0,189.0,484.0,342.0
3,4,372.0,302.0,169.0,114.0,187.0,196.0,71.043478,96.000000,147.0,198.0,436.0,333.0
4,5,375.0,262.0,208.0,188.0,214.0,168.0,71.043478,114.033333,112.0,217.0,473.0,316.0


In [11]:
df_ml_ready.shape
df_ml_ready.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Day        31 non-null     int64  
 1   January    31 non-null     float64
 2   February   31 non-null     float64
 3   March      31 non-null     float64
 4   April      31 non-null     float64
 5   May        31 non-null     float64
 6   June       31 non-null     float64
 7   July       31 non-null     float64
 8   August     31 non-null     float64
 9   September  31 non-null     float64
 10  October    31 non-null     float64
 11  November   31 non-null     float64
 12  December   31 non-null     float64
dtypes: float64(12), int64(1)
memory usage: 3.3 KB
